<a href="https://colab.research.google.com/github/yudithvega-art/bhm-mrsa-indonesia/blob/main/Exposure_Dose_and_Sensitivity_16072026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D


# =============================================================================
# CONFIGURATION
# =============================================================================

MATRIX_ORDER = ["Milk", "Water", "Bioaerosol", "EnvDust"]

MATRIX_COLORS = {
    "Milk": "#D55E00", "Water": "#0072B2",
    "Bioaerosol": "#009E73", "EnvDust": "#CC79A7",
}

# Default exposure factor and dose unit per matrix
EXPOSURE_DEFAULT = {
    "Milk":       {"factor": 1.0,  "unit": "CFU/event", "vlabel": "1.0 mL/event"},
    "Water":      {"factor": 1.0,  "unit": "CFU/event", "vlabel": "1.0 mL/event"},
    "Bioaerosol": {"factor": 0.48, "unit": "CFU/hour",  "vlabel": "0.48 m3/hour"},
    "EnvDust":    {"factor": 0.01, "unit": "CFU/event", "vlabel": "0.01 g/event"},
}

# Sensitivity scenarios: (low, default, conservative)
SENSITIVITY_FACTORS = {
    "Milk":       (0.1,   1.0,  5.0),
    "Water":      (0.1,   1.0,  5.0),
    "Bioaerosol": (0.24,  0.48, 0.96),
    "EnvDust":    (0.001, 0.01, 0.1),
}
SCENARIO_LABELS = ["Low", "Default", "Conservative"]

mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 160, "savefig.bbox": "tight",
})

# Global lists for consistent plotting
ALL_FARMS = [f"F-{i:02d}" for i in range(1, 11)]
PANEL_LABELS = ['A', 'B', 'C', 'D']

def farm_key(f):
    return int(f.split("-")[1])


# =============================================================================
# LOAD POSTERIOR + RECONSTRUCT APPROXIMATE SAMPLES
# =============================================================================

def load_posterior(csv_path):
    """Load posterior farm means; approximate posterior sd on log10 scale."""
    df = pd.read_csv(csv_path)
    df["sd_log10"] = (df["q975"] - df["q025"]) / (2 * 1.96)
    return df


def mc_concentration_samples(row, n_mc, rng):
    """Monte Carlo draws of log10 concentration for one farm (Normal approx)."""
    return rng.normal(row["median"], max(row["sd_log10"], 1e-6), n_mc)


# =============================================================================
# EXPOSURE DOSE
# =============================================================================

def compute_exposure_dose(post, n_mc=10000, seed=42):
    """Posterior exposure dose per farm at the default exposure factor."""
    rng = np.random.default_rng(seed)
    rows = []
    for matrix in MATRIX_ORDER:
        cfg = EXPOSURE_DEFAULT[matrix]
        sub = post[post["matrix"] == matrix]
        for _, r in sub.iterrows():
            log_c = mc_concentration_samples(r, n_mc, rng)
            dose = (10 ** log_c) * cfg["factor"]
            rows.append({
                "matrix": matrix, "farm": r["farm"],
                "unit": cfg["unit"], "factor": cfg["factor"],
                "dose_median": float(np.median(dose)),
                "dose_q025": float(np.percentile(dose, 2.5)),
                "dose_q975": float(np.percentile(dose, 97.5)),
                "dose_mean": float(np.mean(dose)),
            })
    return pd.DataFrame(rows)


# =============================================================================
# SENSITIVITY ANALYSIS
# =============================================================================

def sensitivity_analysis(post, n_mc=10000, seed=7):
    """Per-matrix exposure dose under low / default / conservative factors."""
    rng = np.random.default_rng(seed)
    rows = []
    delta_rows = []

    for matrix in MATRIX_ORDER:
        low, default, high = SENSITIVITY_FACTORS[matrix]
        unit = EXPOSURE_DEFAULT[matrix]["unit"]
        sub = post[post["matrix"] == matrix]

        farm_default_med, farm_low_med, farm_high_med = [], [], []
        for _, r in sub.iterrows():
            log_c = mc_concentration_samples(r, n_mc, rng)
            c = 10 ** log_c
            for factor, label in zip((low, default, high), SCENARIO_LABELS):
                dose = c * factor
                med = float(np.median(dose))
                rows.append({
                    "matrix": matrix, "farm": r["farm"], "scenario": label,
                    "factor": factor, "unit": unit,
                    "dose_median": med,
                    "dose_q025": float(np.percentile(dose, 2.5)),
                    "dose_q975": float(np.percentile(dose, 97.5)),
                })
                if label == "Default":
                    farm_default_med.append(med)
                elif label == "Low":
                    farm_low_med.append(med)
                else:
                    farm_high_med.append(med)

        farm_default_med = np.array(farm_default_med)
        farm_low_med = np.array(farm_low_med)
        farm_high_med = np.array(farm_high_med)
        # Delta = (high - low) / default, averaged across farms, as percentage
        delta = np.mean((farm_high_med - farm_low_med) / farm_default_med) * 100
        delta_rows.append({
            "matrix": matrix, "unit": unit,
            "low_factor": low, "default_factor": default, "high_factor": high,
            "delta_percent": round(float(delta), 1),
            "default_median_min": float(farm_default_med.min()),
            "default_median_max": float(farm_default_med.max()),
        })

    return pd.DataFrame(rows), pd.DataFrame(delta_rows)


# =============================================================================
# PLOTS
# =============================================================================

def plot_exposure_dose(dose_df, outpath):
    """Forest plot of per-farm exposure dose, one panel per matrix."""
    fig, axes = plt.subplots(1, 4, figsize=(17, 6.5))

    for idx, (ax, matrix) in enumerate(zip(axes, MATRIX_ORDER)):
        cfg = EXPOSURE_DEFAULT[matrix]
        sub = dose_df[dose_df["matrix"] == matrix].copy()

        # Order farms from F-01 to F-10 for existing farms
        farms_with_data = [f for f in ALL_FARMS if f in sub['farm'].values]
        sub['farm_order'] = pd.Categorical(sub['farm'], categories=farms_with_data, ordered=True)
        sub = sub.sort_values('farm_order', ascending=True)
        farms = sub["farm"].tolist()

        y = np.arange(len(farms))
        color = MATRIX_COLORS[matrix]

        central = np.maximum(sub["dose_median"].values, 1e-9)
        lo = np.maximum(sub["dose_q025"].values, 1e-9)
        hi = np.maximum(sub["dose_q975"].values, 1e-9)

        ax.errorbar(central, y, xerr=[central - lo, hi - central],
                    fmt="o", color=color, ecolor=color, alpha=0.85,
                    markersize=9, capsize=4, linewidth=1.6,
                    markeredgecolor="white", markeredgewidth=1.3)
        # mean marker (arithmetic, important for QMRA)
        ax.plot(sub["dose_mean"].values, y, "x", color="black",
                markersize=6, markeredgewidth=1.3, alpha=0.8, zorder=5)

        ax.set_yticks(y)
        ax.set_yticklabels(farms, fontsize=10)
        ax.set_xscale("log")
        ax.set_xlabel(f"Exposure dose ({cfg['unit']})  ·  log scale")
        ax.grid(axis="x", alpha=0.25, which="major")
        ax.grid(axis="x", alpha=0.10, which="minor", linestyle=":")
        ax.invert_yaxis()
        ax.text(-0.1, 1.05, PANEL_LABELS[idx], transform=ax.transAxes, fontsize=14, fontweight='bold', va='top', ha='right')

    legend = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#555",
               markersize=9, markeredgecolor="white",
               label="Posterior median + 95% CrI"),
        Line2D([0], [0], marker="x", color="black", markersize=6,
               linestyle="", label="Arithmetic mean"),
    ]
    fig.legend(handles=legend, loc="lower center", ncol=2, frameon=False,
               fontsize=9.5, bbox_to_anchor=(0.5, -0.03))
    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()


def plot_sensitivity(scen_df, delta_df, outpath):
    """Median per-event exposure dose across the three volume scenarios
    (low, default, conservative), one panel per matrix. The line traces the
    median exposure dose (across sampling points within the matrix) at each
    scenario, with the numeric value labeled at each point and the sensitivity
    range (Delta) annotated per panel."""
    fig, axes = plt.subplots(1, 4, figsize=(17, 6.0))
    vol_unit = {"Milk": "mL", "Water": "mL", "Bioaerosol": "m\u00b3/h", "EnvDust": "g"}

    def fmt(v):
        if v < 0.01:
            return f"{v:.1e}"
        if v < 10:
            return f"{v:.2f}"
        if v < 1000:
            return f"{v:.0f}"
        return f"{v:.2g}"

    for idx, (ax, matrix) in enumerate(zip(axes, MATRIX_ORDER)):
        color = MATRIX_COLORS[matrix]
        unit = EXPOSURE_DEFAULT[matrix]["unit"]
        low, default, high = SENSITIVITY_FACTORS[matrix]
        sub = scen_df[scen_df["matrix"] == matrix]

        # median across sampling points (farms) at each scenario
        med = {lab: sub[sub["scenario"] == lab]["dose_median"].median()
               for lab in SCENARIO_LABELS}
        x = np.array([0, 1, 2])
        yvals = [med["Low"], med["Default"], med["Conservative"]]

        ax.plot(x, yvals, "-", color=color, lw=2.6, zorder=3,
                solid_capstyle="round")
        ax.plot(x, yvals, "o", color=color, markersize=11, zorder=4,
                markeredgecolor="white", markeredgewidth=1.5)

        # value labels
        for xi, yi in zip(x, yvals):
            ax.annotate(fmt(yi), (xi, yi), textcoords="offset points",
                        xytext=(0, 12), ha="center", fontsize=9.5,
                        fontweight="bold", color="#333")

        ax.set_xticks(x)
        ax.set_xticklabels([
            f"Low\n{low:g} {vol_unit[matrix]}",
            f"Default\n{default:g} {vol_unit[matrix]}",
            f"Conserv.\n{high:g} {vol_unit[matrix]}"], fontsize=9.5)
        ax.set_xlim(-0.4, 2.4)
        ax.set_yscale("log")
        ax.set_ylabel(f"Median exposure dose ({unit})  ·  log scale", fontsize=9.5)
        # headroom for labels
        ax.set_ylim(min(yvals) / 3.0, max(yvals) * 3.0)
        ax.grid(axis="y", alpha=0.25, which="major")
        ax.grid(axis="y", alpha=0.10, which="minor", linestyle=":")
        ax.text(-0.1, 1.05, PANEL_LABELS[idx], transform=ax.transAxes, fontsize=14, fontweight='bold', va='top', ha='right')

    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()


# =============================================================================
# MAIN
# =============================================================================

# =============================================================================
# RIDGELINE EXPOSURE DOSE DISTRIBUTION (added) - Renamed from plot_exposure_dose to plot_exposure_ridgeline
# =============================================================================

def plot_exposure_ridgeline(post, outpath, n_mc=10000, seed=123):
    """Ridgeline density plot of the per-event exposure dose distribution
    per farm, one panel per matrix. Reads clearly as an exposure dose
    distribution rather than a dose-response or forest plot."""
    from scipy.stats import gaussian_kde

    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(1, 4, figsize=(17, 8.5))

    for idx, (ax, matrix) in enumerate(zip(axes, MATRIX_ORDER)):
        cfg = EXPOSURE_DEFAULT[matrix]
        sub = post[post["matrix"] == matrix].copy()

        # exposure dose samples per farm (log10) at default factor
        dose_log = {}
        for _, r in sub.iterrows():
            log_c = mc_concentration_samples(r, n_mc, rng)
            dose_log[r["farm"]] = log_c + np.log10(cfg["factor"])

        # order farms from F-01 to F-10 for existing farms
        farms = [f for f in ALL_FARMS if f in dose_log]

        # shared x-grid per panel
        all_vals = np.concatenate(list(dose_log.values())) if dose_log else []
        if all_vals.size > 0:
            x_lo, x_hi = np.percentile(all_vals, [0.5, 99.5])
        else:
            x_lo, x_hi = -1, 1 # Default range if no data

        grid = np.linspace(x_lo - 0.3, x_hi + 0.3, 300)
        x_lin = 10 ** grid
        offset = 1.25
        color = MATRIX_COLORS[matrix]
        y_ticks = []

        for i, farm in enumerate(farms):
            d = dose_log[farm]
            try:
                kde = gaussian_kde(d, bw_method=0.35)
                dens = kde(grid)
                dens = dens / dens.max() * 0.95
            except Exception:
                dens = np.zeros_like(grid)
            base = (len(farms) - i - 1) * offset
            y_ticks.append(base + 0.4)
            ax.fill_between(x_lin, base, base + dens, color=color, alpha=0.42,
                            edgecolor=color, linewidth=1.1, zorder=3)
            med_d = 10 ** np.median(d)
            ax.plot([med_d, med_d], [base, base + 0.65], color="black",
                    lw=1.5, alpha=0.85, zorder=5)
            for pc in (5, 95):
                pv = 10 ** np.percentile(d, pc)
                ax.plot([pv, pv], [base, base + 0.18], color="#333",
                        lw=1.6, alpha=0.75, zorder=4)

        ax.set_yticks(y_ticks)
        ax.set_yticklabels(farms, fontsize=9.5)
        ax.set_xscale("log")
        ax.set_xlabel(f"Exposure dose ({cfg['unit']})  ·  log scale")
        ax.set_ylim(-0.3, len(farms) * offset + 0.15)
        ax.grid(axis="x", alpha=0.25, which="major")
        ax.grid(axis="x", alpha=0.10, which="minor", linestyle=":")
        ax.spines["left"].set_visible(False)
        ax.tick_params(axis="y", length=0)
        ax.text(-0.1, 1.05, PANEL_LABELS[idx], transform=ax.transAxes, fontsize=14, fontweight='bold', va='top', ha='right')

    legend = [
        Line2D([0], [0], color="#999", lw=10, alpha=0.42,
               label="Exposure dose density per farm"),
        Line2D([0], [0], color="black", lw=1.5, label="Median"),
        Line2D([0], [0], color="#333", lw=1.6, alpha=0.75,
               label="5th and 95th percentiles"),
    ]
    fig.legend(handles=legend, loc="lower center", ncol=3, frameon=False,
               fontsize=9.5, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()


def plot_hotspot_grid(dose_df, post, outpath):
    """Ten-farm by four-matrix grid for hotspot identification. Each cell shows
    the default per-event exposure dose per farm and matrix, colored within
    each matrix column on a log scale (darker = higher). The single highest
    farm per matrix (the intervention hotspot) is outlined. Because exposure
    dose is a fixed per-matrix multiple of concentration, the hotspot ranking
    is identical whether judged by concentration or by exposure dose."""
    from matplotlib.colors import LogNorm
    from matplotlib.patches import Rectangle

    farms = [f"F-{i:02d}" for i in range(1, 11)]

    grid = pd.DataFrame(index=farms, columns=MATRIX_ORDER, dtype=float)
    for _, r in dose_df.iterrows():
        grid.loc[r["farm"], r["matrix"]] = r["dose_median"]

    fig, ax = plt.subplots(figsize=(10, 8.2))

    cmap_by_matrix = {"Milk": "Oranges", "Water": "Blues",
                      "Bioaerosol": "Greens", "EnvDust": "RdPu"}
    rgba = np.ones((len(farms), len(MATRIX_ORDER), 4))
    hotspot = {}
    for j, matrix in enumerate(MATRIX_ORDER):
        col = grid[matrix].values.astype(float)
        finite = col[np.isfinite(col)]
        if len(finite) == 0:
            continue
        vmin, vmax = np.nanmin(finite), np.nanmax(finite)
        norm = LogNorm(vmin=max(vmin, 1e-9), vmax=max(vmax, vmin * 1.001))
        base = plt.cm.get_cmap(cmap_by_matrix[matrix])
        for i in range(len(farms)):
            v = col[i]
            if np.isfinite(v):
                rgba[i, j] = base(0.20 + 0.75 * norm(v))
            else:
                rgba[i, j] = (0.96, 0.96, 0.96, 1.0)
        hotspot[matrix] = int(np.nanargmax(col))

    ax.imshow(rgba, aspect="auto")

    unit_map = {m: EXPOSURE_DEFAULT[m]["unit"] for m in MATRIX_ORDER}
    ax.set_xticks(range(len(MATRIX_ORDER)))
    ax.set_xticklabels([f"{m}\n({unit_map[m]})" for m in MATRIX_ORDER],
                       fontsize=10.5)
    ax.set_yticks(range(len(farms)))
    ax.set_yticklabels(farms, fontsize=10)
    ax.set_xlabel("Matrix", fontsize=11, labelpad=10)
    ax.set_ylabel("Farm", fontsize=11)

    def fmt(v):
        if v < 0.01:
            return f"{v:.1e}"
        if v < 10:
            return f"{v:.2f}"
        if v < 1000:
            return f"{v:.0f}"
        return f"{v:.2g}"

    for i in range(len(farms)):
        for j, matrix in enumerate(MATRIX_ORDER):
            v = grid.iloc[i, j]
            if not np.isfinite(v):
                ax.text(j, i, "\u2014", ha="center", va="center",
                        fontsize=10, color="#bbb")
                continue
            lum = np.mean(rgba[i, j, :3])
            ax.text(j, i, fmt(v), ha="center", va="center", fontsize=9,
                    color="white" if lum < 0.5 else "#222")

    for j, matrix in enumerate(MATRIX_ORDER):
        if matrix in hotspot:
            i = hotspot[matrix]
            ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                    edgecolor="black", linewidth=2.6, zorder=6))

    ax.set_xticks(np.arange(-0.5, len(MATRIX_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(farms), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2)
    ax.tick_params(which="minor", length=0)
    ax.tick_params(which="major", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.text(-0.1, 1.05, 'A', transform=ax.transAxes, fontsize=14, fontweight='bold', va='top', ha='right')

    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()


def main(input_csv, output_dir, n_mc):
    out = Path(output_dir)
    out.mkdir(exist_ok=True, parents=True)

    post = load_posterior(input_csv)

    # Exposure dose at default factors
    dose_df = compute_exposure_dose(post, n_mc=n_mc)
    dose_df.to_csv(out / "exposure_dose_per_farm.csv", index=False)

    # Sensitivity analysis
    scen_df, delta_df = sensitivity_analysis(post, n_mc=n_mc)
    scen_df.to_csv(out / "exposure_sensitivity_scenarios.csv", index=False)
    delta_df.to_csv(out / "exposure_sensitivity.csv", index=False)

    # Figures
    plot_exposure_ridgeline(post, out / "fig_exposure_dose.png", n_mc=n_mc)
    plot_sensitivity(scen_df, delta_df, out / "fig_exposure_sensitivity.png")
    plot_hotspot_grid(dose_df, post, out / "fig_exposure_hotspot_grid.png")

    # Terminal report
    print("=== Per-event exposure dose (default factors) ===")
    for matrix in MATRIX_ORDER:
        cfg = EXPOSURE_DEFAULT[matrix]
        sub = dose_df[dose_df["matrix"] == matrix].sort_values(
            "dose_median", ascending=False)
        print(f"\n{matrix}  ({cfg['vlabel']}, {cfg['unit']}):")
        for _, r in sub.iterrows():
            print(f"  {r['farm']}: median {r['dose_median']:.3g}  "
                  f"[{r['dose_q025']:.3g}, {r['dose_q975']:.3g}]  "
                  f"mean {r['dose_mean']:.3g}")

    print("\n=== Sensitivity (Delta = relative change, % of default) ===")
    for _, r in delta_df.iterrows():
        print(f"  {r['matrix']:<12} Delta = {r['delta_percent']:.0f}%   "
              f"default median range {r['default_median_min']:.3g} to "
              f"{r['default_median_max']:.3g} {r['unit']}")

    print(f"\nSaved outputs in: {out.resolve()}")


### Run Exposure Dose Analysis

This cell executes the `main` function from the Python script to perform the exposure dose estimation and sensitivity analysis. It takes `bhm_full_mu_F.csv` as input, saves outputs to `exposure_output` directory, and uses `10000` Monte Carlo simulations.

In [8]:
from pathlib import Path

input_file = '/content/bhm_full_mu_F.csv'
output_directory = './exposure_output'
n_mc_simulations = 10000

# Create the output directory if it doesn't exist
Path(output_directory).mkdir(exist_ok=True, parents=True)

# Call the main function directly now that it's defined in the global scope
main(input_file, output_directory, n_mc_simulations)

/tmp/ipykernel_1977/2565158864.py:382: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  base = plt.cm.get_cmap(cmap_by_matrix[matrix])


=== Per-event exposure dose (default factors) ===

Milk  (1.0 mL/event, CFU/event):
  F-10: median 843  [30.9, 2.37e+04]  mean 3.78e+03
  F-09: median 520  [24.3, 1.16e+04]  mean 1.9e+03
  F-08: median 487  [24.6, 9.84e+03]  mean 1.61e+03
  F-05: median 153  [9.59, 2.58e+03]  mean 428
  F-07: median 50.9  [2.22, 1.19e+03]  mean 179
  F-02: median 15.3  [0.315, 777]  mean 113

Water  (1.0 mL/event, CFU/event):
  F-02: median 1.58  [0.28, 9.03]  mean 2.34
  F-05: median 1.21  [0.24, 6.09]  mean 1.72
  F-08: median 0.983  [0.218, 4.33]  mean 1.32
  F-09: median 0.781  [0.151, 3.96]  mean 1.09
  F-06: median 0.758  [0.17, 3.61]  mean 1.04
  F-07: median 0.438  [0.0303, 6.75]  mean 1.16
  F-04: median 0.424  [0.027, 6.49]  mean 1.11
  F-10: median 0.343  [0.0226, 5.14]  mean 0.903
  F-03: median 0.332  [0.0232, 5.04]  mean 0.872

Bioaerosol  (0.48 m3/hour, CFU/hour):
  F-06: median 31.5  [13.8, 71.2]  mean 34.4
  F-03: median 28.7  [13.2, 62.6]  mean 31.1
  F-02: median 28.4  [13, 63.4]  me